In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from config.settings import DATA_DIR, OUTPUT_DIR
from src.utils.data_store import load
from src.analysis.lda_model import build_vectorizer, build_lda, get_top_words
from src.analysis.trend_analysis import compute_topic_trends, classify_topics
from src.visualization.timeline import plot_publications_per_year, plot_hot_cold_topics
from src.visualization.lda_vis import save_interactive_lda
df = load("scopus_devops", DATA_DIR)
texts = df["Abstract_clean"].dropna().tolist()
dtm, vectorizer = build_vectorizer(texts)
model, doc_topic_matrix = build_lda(dtm, k=30, alpha=0.1, beta=0.01, passes=20)

In [ ]:
years = pd.to_datetime(df["Date"], errors="coerce").dt.year.dropna().astype(int).tolist()
years = years[:len(doc_topic_matrix)]
trends = compute_topic_trends(doc_topic_matrix, years)
classified = classify_topics(trends)
top_words = get_top_words(model, vectorizer)
result = classified.merge(top_words, on="topic_id")
result.sort_values("slope", ascending=False)

In [ ]:
hot = result[result["trend_class"] == "hot"].nlargest(10, "slope")
print("HOT TOPICS:")
print(hot[["slope", "p_value", "top_words"]].to_string())

In [ ]:
cold = result[result["trend_class"] == "cold"].nsmallest(10, "slope")
print("COLD TOPICS:")
print(cold[["slope", "p_value", "top_words"]].to_string())

In [ ]:
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
from IPython.display import Image
path = plot_hot_cold_topics(classified, top_words, output_path=f"{OUTPUT_DIR}/hot_cold.png")
Image(path)
save_interactive_lda(model, dtm, vectorizer, output_path=f"{OUTPUT_DIR}/lda_interactive.html")